# Diffusion World Model on Kaggle — Push-T (`swm/PushT-v1`)

Trains and evaluates the **SD-VAE (4x28x28) + Rectified Flow / Diffusion Forcing DiT** world model on **Push-T**.

> **Kaggle Settings Required:**
> - **Accelerator:** GPU (T4 x2 or P100 16 GB)
> - **Internet:** ON

In [ ]:
import datetime
import time

def utc_now_str():
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

def fmt_duration(seconds):
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

NOTEBOOK_START_UTC = utc_now_str()
NOTEBOOK_START_MONO = time.monotonic()
STAGE_TIMINGS = {}
print(f"[{NOTEBOOK_START_UTC}] Notebook started")

# === 1. User Configuration ===
GITHUB_REPO_URL = "https://github.com/hanriman/diffusion-wm.git"
BRANCH = "main"

ENV_NAME = "pusht"
SOLVER = "gd"       # "gd" (GradientSolver, recommended) or "cem" (NanoWM-style parallel CEMSolver)
SEED = 42

SKIP_TRAINING = True        # True: attach committed rightfel/pusht kernel output for fast eval + demo; False to train
IMG_SIZE = 128              # 128x128 -> 64 DiT tokens/frame (3x faster than 224x224, crisp T-block)
PRECOMPUTE_VAE_LATENTS = True  # ~1.2 GB at 128x128 on pusht-5k; skipped automatically when SKIP_TRAINING=True
MAX_EPOCHS = 15
BATCH_SIZE = 32             # 32 fits T4 15GB VRAM comfortably with num_steps=8 (528 tokens/clip)
LIMIT_TRAIN_BATCHES = 2500  # 37.5k steps (~2.5-3.5h on T4 with num_steps=8 precomputed latents)
LIMIT_VAL_BATCHES = 100     # validation batches per epoch (set None for full val set)
TOTAL_TRANSITIONS = None    # optional dataset transition cap (e.g. 200000) or None
SAVE_EVERY_N_EPOCHS = 0     # 0 keeps only the final *_object.ckpt to prevent /kaggle/working disk full
NUM_EVAL = 10               # set 50 for full benchmark eval
EVAL_NUM_SAMPLES = 32 if SOLVER == "gd" else 256
RUN_LIVE_DEMO = True        # True: renders live GPU sliders in interactive Kaggle/Jupyter sessions
EVAL_HORIZON = None         # optional int override (e.g. 3, 4, 6) or None to use eval yaml default
EVAL_PROPRIO_COST_WEIGHT = None  # optional float override (e.g. 0.05) or None to use eval yaml default
VIZ_EPISODES = None               # optional list of episode indices (e.g. [2, 4, 7]) or None

REPO_DIR = "/kaggle/working/diffusion-wm"
DATA_DIR = "/tmp/hjepa_data"
CKPT_DIR = f"/kaggle/working/ckpts/{ENV_NAME}/{ENV_NAME}_dwm/seed{SEED}"
KAGGLE_INPUT_DATA_DIR = ""  # Option A: leave empty to auto-download from HuggingFace
PRETRAINED_CKPT_DIR = ""    # optional explicit path; auto-discovered under /kaggle/input when SKIP_TRAINING=True


In [ ]:
%%bash
nvidia-smi
df -h /kaggle/working /tmp
apt-get update -qq && apt-get install -y -qq zstd zip > /dev/null || true
pip install -q uv
which uv && uv --version

In [ ]:
import os
import subprocess

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["HJEPA_HOME"] = DATA_DIR
os.environ["MUJOCO_GL"] = "egl"
os.environ["MPLBACKEND"] = "Agg"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

%cd {REPO_DIR}

In [ ]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
uv venv --python 3.11
uv sync
uv run python -c "import torch, cv2, pymunk, mujoco, ogbench; print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), 'cv2:', cv2.__version__)"

### 2. Download Evaluation Tasks & PUSHT Dataset (Option A)
- Downloads the 50 fixed evaluation tasks into `diffusion_wm/assets/eval_trajs/`.
- Downloads `pusht` HDF5 train/val datasets into `DATA_DIR` (`/tmp/hjepa_data`).

In [ ]:
t0_data = time.monotonic()
print(f"[{utc_now_str()}] Downloading eval tasks" + (" (skipping HDF5 training data for eval-only run)..." if SKIP_TRAINING else " and dataset..."))
import glob
import subprocess

subprocess.run(["bash", "scripts/download_eval_trajs.sh"], check=True)

if not SKIP_TRAINING:
    if KAGGLE_INPUT_DATA_DIR and os.path.isdir(KAGGLE_INPUT_DATA_DIR):
        for h5_file in glob.glob(os.path.join(KAGGLE_INPUT_DATA_DIR, "*.h5")):
            dst = os.path.join(DATA_DIR, os.path.basename(h5_file))
            if not os.path.exists(dst):
                os.symlink(h5_file, dst)
        print("Linked HDF5 files from Kaggle Input:", os.listdir(DATA_DIR))
    else:
        subprocess.run(["bash", "scripts/download_datasets.sh", ENV_NAME], check=True)
STAGE_TIMINGS["data_download_sec"] = time.monotonic() - t0_data
print(f"[{utc_now_str()}] Data prep finished in {fmt_duration(STAGE_TIMINGS['data_download_sec'])}")


### 3. (Optional) Precompute SD-VAE Latents
Skipped when `PRECOMPUTE_VAE_LATENTS = False`.

In [ ]:
if PRECOMPUTE_VAE_LATENTS and not SKIP_TRAINING:
    import shutil
    h5_files = ["pusht_expert_train.h5", "pusht_expert_val.h5"]
    for fname in h5_files:
        fpath = os.path.join(DATA_DIR, fname)
        if os.path.islink(fpath):
            target = os.path.realpath(fpath)
            os.unlink(fpath)
            shutil.copy2(target, fpath)
        subprocess.run([
            "uv", "run", "python", "scripts/precompute_vae_latents.py",
            fpath, "--batch-size", "100", "--img-size", str(IMG_SIZE)
        ], check=True)
elif SKIP_TRAINING:
    print("Skipping VAE latent precomputation (SKIP_TRAINING=True).")


### 4. Train Diffusion World Model
Uses `trainer.precision=16-mixed` and `trainer.devices=1` for Kaggle T4/P100 compatibility, and saves checkpoints + `normalizer.pt` to `/kaggle/working/ckpts/...`.

In [ ]:
t0_train = time.monotonic()
print(f"[{utc_now_str()}] Starting training stage...")
import glob
import shutil

ckpt_filename = f"{ENV_NAME}_dwm_object.ckpt"
expected_ckpt = os.path.join(CKPT_DIR, ckpt_filename)
expected_norm = os.path.join(CKPT_DIR, "normalizer.pt")

src_ckpt_dir = PRETRAINED_CKPT_DIR
if not src_ckpt_dir and SKIP_TRAINING and not (os.path.exists(expected_ckpt) and os.path.exists(expected_norm)):
    matches = sorted(glob.glob(f"/kaggle/input/**/{ckpt_filename}", recursive=True))
    if matches:
        src_ckpt_dir = os.path.dirname(matches[0])
        print("Auto-discovered Kaggle Input checkpoint dir:", src_ckpt_dir)

if src_ckpt_dir and os.path.isdir(src_ckpt_dir):
    for fname in ("pusht_dwm_object.ckpt", "normalizer.pt", "config.yaml"):
        src = os.path.join(src_ckpt_dir, fname)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(CKPT_DIR, fname))
    logs_src = os.path.join(src_ckpt_dir, "lightning_logs")
    logs_dst = os.path.join(CKPT_DIR, "lightning_logs")
    if os.path.isdir(logs_src) and not os.path.exists(logs_dst):
        shutil.copytree(logs_src, logs_dst)
    print("Copied pretrained checkpoint artifacts from", src_ckpt_dir)

if os.path.exists(expected_ckpt) and os.path.exists(expected_norm):
    print("Found existing checkpoint, skipping training:", expected_ckpt)
elif SKIP_TRAINING:
    raise FileNotFoundError(
        f"SKIP_TRAINING=True, but {expected_ckpt} and {expected_norm} were not found. "
        "Attach your committed rightfel/pusht notebook output in Kaggle or set PRETRAINED_CKPT_DIR."
    )
else:
    train_cfg = f"{ENV_NAME}_dwm"
    cmd = [
        "uv", "run", "python", "-m", "diffusion_wm.train",
        "--config-name", train_cfg,
        f"seed={SEED}",
        f"img_size={IMG_SIZE}",
        f"cache_dir={DATA_DIR}",
        f"run_dir={CKPT_DIR}",
        f"save_every_n_epochs={SAVE_EVERY_N_EPOCHS}",
        "trainer.precision=16-mixed",
        "trainer.devices=1",
        f"trainer.max_epochs={MAX_EPOCHS}",
        f"loader.batch_size={BATCH_SIZE}",
        "loader.num_workers=2",
        "planning_eval.enabled=false",
    ]
    if LIMIT_TRAIN_BATCHES is not None:
        cmd.append(f"trainer.limit_train_batches={LIMIT_TRAIN_BATCHES}")
    if LIMIT_VAL_BATCHES is not None:
        cmd.append(f"trainer.limit_val_batches={LIMIT_VAL_BATCHES}")
    if TOTAL_TRANSITIONS is not None:
        cmd.append(f"data.dataset.total_transitions={TOTAL_TRANSITIONS}")
    if PRECOMPUTE_VAE_LATENTS:
        cmd.append("data.dataset.keys_to_load=[vae_latents,action,proprio,state]")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
STAGE_TIMINGS["train_sec"] = time.monotonic() - t0_train
print(f"[{utc_now_str()}] Training stage finished in {fmt_duration(STAGE_TIMINGS['train_sec'])}")


### 5. Evaluate Closed-Loop MPC Planning
Runs `diffusion_wm.eval` using the trained checkpoint and prints `metrics.yaml`.

In [ ]:
t0_eval = time.monotonic()
print(f"[{utc_now_str()}] Starting MPC evaluation...")
ckpt_path = os.path.join(CKPT_DIR, f"{ENV_NAME}_dwm_object.ckpt")
eval_cfg = f"{ENV_NAME}_{SOLVER}"
eval_out_dir = f"eval_{SOLVER}"

eval_cmd = [
    "uv", "run", "python", "-m", "diffusion_wm.eval",
    "--config-name", eval_cfg,
    f"seed={SEED}",
    f"cache_dir={DATA_DIR}",
    f"policy={ckpt_path}",
    f"eval.img_size={IMG_SIZE}",
    f"eval.num_eval={NUM_EVAL}",
    f"solver.num_samples={EVAL_NUM_SAMPLES}",
    f"output.dir={eval_out_dir}",
]
if EVAL_HORIZON is not None:
    eval_cmd.append(f"plan_config.horizon={EVAL_HORIZON}")
if EVAL_PROPRIO_COST_WEIGHT is not None:
    eval_cmd.append(f"model_overrides.proprio_cost_weight={EVAL_PROPRIO_COST_WEIGHT}")
print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)

metrics_file = os.path.join(CKPT_DIR, eval_out_dir, "metrics.yaml")
with open(metrics_file) as f:
    print("\n=== Evaluation Metrics ===\n" + f.read())
STAGE_TIMINGS["eval_sec"] = time.monotonic() - t0_eval
print(f"[{utc_now_str()}] Evaluation finished in {fmt_duration(STAGE_TIMINGS['eval_sec'])}")


### 6. Generate & Display Presentation Visuals
Generates training loss curves, open-loop **World Model Dreams** (`Ground Truth` vs `SD-VAE Recon` vs `Diffusion WM`), evaluation summary plots, and side-by-side **Closed-Loop MPC Rollout GIFs** for your presentation.

In [ ]:
t0_viz = time.monotonic()
print(f"[{utc_now_str()}] Generating presentation visuals...")
import glob
import shutil
from IPython.display import FileLink, Image as IPyImage, display

viz_dir = os.path.join(CKPT_DIR, "visualizations")
common_overrides = [
    "--seed", str(SEED),
    "--img-size", str(IMG_SIZE),
    "--num-samples", str(EVAL_NUM_SAMPLES),
]
if EVAL_HORIZON is not None:
    common_overrides.extend(["--horizon", str(EVAL_HORIZON)])
if EVAL_PROPRIO_COST_WEIGHT is not None:
    common_overrides.extend(["--proprio-cost-weight", str(EVAL_PROPRIO_COST_WEIGHT)])
if VIZ_EPISODES:
    common_overrides.append("--episodes")
    common_overrides.extend(str(e) for e in VIZ_EPISODES)

viz_cmd = [
    "uv", "run", "python", "-m", "diffusion_wm.visualize",
    "--eval-config", eval_cfg,
    "--policy", ckpt_path,
    "--run-dir", CKPT_DIR,
    "--eval-dir", os.path.join(CKPT_DIR, eval_out_dir),
    "--out-dir", viz_dir,
    "--cache-dir", DATA_DIR,
    "--num-dreams", "2",
    "--num-videos", "1",
    "--max-rollout-steps", "45",
    *common_overrides,
]
print("Running:", " ".join(viz_cmd))
subprocess.run(viz_cmd, check=True)

demo_html = os.path.join(viz_dir, "diffusion_wm_demo.html")
demo_cmd = [
    "uv", "run", "python", "-m", "diffusion_wm.demo",
    "--eval-config", eval_cfg,
    "--policy", ckpt_path,
    "--eval-dir", os.path.join(CKPT_DIR, eval_out_dir),
    "--out-path", demo_html,
    "--cache-dir", DATA_DIR,
    "--num-ode-steps", "8",
    *common_overrides,
]
print("Running:", " ".join(demo_cmd))
subprocess.run(demo_cmd, check=True)
local_demo_rel = os.path.join("artifacts", "diffusion_wm_demo.html")
local_demo_abs = os.path.join(REPO_DIR, local_demo_rel)
os.makedirs(os.path.dirname(local_demo_abs), exist_ok=True)
shutil.copy2(demo_html, local_demo_abs)
print("Interactive lecture demo saved to:", demo_html)
display(FileLink(local_demo_rel))

for img_path in sorted(glob.glob(os.path.join(viz_dir, "*.png")) + glob.glob(os.path.join(viz_dir, "*.gif"))):
    print("\n>>>", os.path.basename(img_path))
    display(IPyImage(filename=img_path))
STAGE_TIMINGS["visualize_sec"] = time.monotonic() - t0_viz
print(f"[{utc_now_str()}] Visualizations finished in {fmt_duration(STAGE_TIMINGS['visualize_sec'])}")


### 6b. (Interactive Only) Live Counterfactual & Diffusion Slider
Set `RUN_LIVE_DEMO = True` in Cell 1 (or here) in an interactive Kaggle/Jupyter session to drag `(action_x, action_y, horizon k, diffusion step tau)` sliders live. Download `diffusion_wm_demo.html` from `pusht_results.zip` for an offline browser demo in class.

In [ ]:
if RUN_LIVE_DEMO:
    import atexit
    import json
    import subprocess
    import ipywidgets as widgets
    from IPython.display import display

    if "_DWM_DEMO_PROC" in globals() and _DWM_DEMO_PROC is not None and _DWM_DEMO_PROC.poll() is None:
        _DWM_DEMO_PROC.terminate()

    _serve_cmd = [
        "uv", "run", "python", "-m", "diffusion_wm.demo",
        "--serve",
        "--eval-config", eval_cfg,
        "--policy", ckpt_path,
        "--cache-dir", DATA_DIR,
        "--img-size", str(IMG_SIZE),
    ]
    if EVAL_HORIZON is not None:
        _serve_cmd.extend(["--horizon", str(EVAL_HORIZON)])
    _DWM_DEMO_PROC = subprocess.Popen(
        _serve_cmd,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )
    atexit.register(lambda: _DWM_DEMO_PROC.poll() is None and _DWM_DEMO_PROC.terminate())

    ready_line = _DWM_DEMO_PROC.stdout.readline()
    if not ready_line:
        raise RuntimeError(f"Demo worker failed to start:\n{_DWM_DEMO_PROC.stderr.read()}")
    meta = json.loads(ready_line)
    h_max = int(meta.get("horizon", 5))
    ep_max = min(9, int(meta.get("max_ep", 9)))
    print(f"Live demo worker ready on {meta.get('device')} (horizon={h_max})")

    _cache = {}
    w_ep = widgets.IntSlider(value=0, min=0, max=ep_max, description="Episode")
    w_ax = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.1, description="Action X")
    w_ay = widgets.FloatSlider(value=-0.5, min=-1.0, max=1.0, step=0.1, description="Action Y")
    w_k = widgets.IntSlider(value=h_max, min=1, max=h_max, description="Horizon k")
    w_steps = widgets.IntSlider(value=8, min=2, max=16, step=2, description="ODE Steps")
    w_tau = widgets.IntSlider(value=8, min=0, max=8, description="Tau Step")
    w_sigma = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.5, description="Noise Sigma")
    view = widgets.HTML()

    def _sync_tau(*_):
        w_tau.max = w_steps.value
        w_tau.value = min(w_tau.value, w_steps.value)
    w_steps.observe(_sync_tau, names="value")

    def _render(*_):
        key = (w_ep.value, round(w_ax.value, 2), round(w_ay.value, 2), w_steps.value, round(w_sigma.value, 2))
        if key not in _cache:
            _DWM_DEMO_PROC.stdin.write(json.dumps({
                "ep": key[0], "ax": key[1], "ay": key[2], "steps": key[3], "sigma": key[4]
            }) + "\n")
            _DWM_DEMO_PROC.stdin.flush()
            resp_line = _DWM_DEMO_PROC.stdout.readline()
            if not resp_line:
                view.value = f"<pre>Worker exited: {_DWM_DEMO_PROC.stderr.read()}</pre>"
                return
            _cache.clear()
            _cache[key] = json.loads(resp_line)

        res = _cache[key]
        k_idx = min(w_k.value - 1, len(res["frames"][0]) - 1)
        tau_idx = min(w_tau.value, len(res["taus"]) - 1)
        tau_val = res["taus"][tau_idx]
        dream_src = res["frames"][tau_idx][k_idx]
        view.value = (
            "<div style='display:flex;gap:14px;align-items:flex-start;font-family:sans-serif'>"
            f"<div style='text-align:center'><div><b>Context (Ep {w_ep.value}, dx={w_ax.value:.1f}, dy={w_ay.value:.1f})</b></div>"
            f"<img src='{res['ctx']}' style='width:224px;border:1px solid #444;border-radius:6px'/></div>"
            f"<div style='text-align:center'><div><b>WM Dream (k=+{k_idx+1}, &tau;={tau_val:.2f})</b></div>"
            f"<img src='{dream_src}' style='width:224px;border:1px solid #444;border-radius:6px'/></div>"
            f"<div style='text-align:center'><div><b>Target Goal</b></div>"
            f"<img src='{res['goal']}' style='width:224px;border:1px solid #444;border-radius:6px'/></div>"
            "</div>"
        )

    for w in (w_ep, w_ax, w_ay, w_k, w_steps, w_tau, w_sigma):
        w.observe(_render, names="value")

    display(widgets.VBox([
        widgets.HBox([w_ep, w_steps, w_sigma]),
        widgets.HBox([w_ax, w_ay]),
        widgets.HBox([w_k, w_tau]),
        view,
    ]))
    _render()
else:
    print("Skipping interactive widget (RUN_LIVE_DEMO=False). Open diffusion_wm_demo.html locally for offline sliders.")


### 7. Package Outputs for Download
Creates `/kaggle/working/pusht_results.zip` containing your checkpoints, normalizer, and evaluation metrics.

In [ ]:
import json

NOTEBOOK_END_UTC = utc_now_str()
total_sec = time.monotonic() - NOTEBOOK_START_MONO
STAGE_TIMINGS["total_sec"] = total_sec

summary_lines = [
    "=== Notebook Run Timing Summary ===",
    f"Environment : {ENV_NAME} (solver={SOLVER}, seed={SEED})",
    f"Start Time  : {NOTEBOOK_START_UTC}",
    f"End Time    : {NOTEBOOK_END_UTC}",
    f"Data Prep   : {fmt_duration(STAGE_TIMINGS.get('data_download_sec', 0))}",
    f"Training    : {fmt_duration(STAGE_TIMINGS.get('train_sec', 0))}",
    f"Evaluation  : {fmt_duration(STAGE_TIMINGS.get('eval_sec', 0))}",
    f"Visuals     : {fmt_duration(STAGE_TIMINGS.get('visualize_sec', 0))}",
    f"Total Time  : {fmt_duration(total_sec)} ({total_sec:.1f}s)",
]
summary_text = "\n".join(summary_lines)
print(summary_text)

timing_txt = os.path.join(CKPT_DIR, "run_timing.txt")
timing_json = os.path.join(CKPT_DIR, "run_timing.json")
with open(timing_txt, "w") as f:
    f.write(summary_text + "\n")
with open(timing_json, "w") as f:
    json.dump({
        "env": ENV_NAME,
        "solver": SOLVER,
        "seed": SEED,
        "start_utc": NOTEBOOK_START_UTC,
        "end_utc": NOTEBOOK_END_UTC,
        "durations_sec": STAGE_TIMINGS,
        "total_hms": fmt_duration(total_sec),
    }, f, indent=2)

zip_path = "/kaggle/working/pusht_results.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
subprocess.run(
    ["zip", "-r", "-q", zip_path, f"/kaggle/working/ckpts/{ENV_NAME}", "-x", "*_epoch_*_object.ckpt"],
    check=True,
)
print("Ready to download:", zip_path, f"({os.path.getsize(zip_path) / (1024**2):.1f} MB)")
